Import Packages 

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import scipy.io
import os
import numpy as np
from scipy.stats import pearsonr
import lime 
import lime.lime_tabular
import numpy as np 
import shap

import pandas as pd
import joblib
import numpy as np
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

C:\Users\yihun\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import warnings
warnings.filterwarnings("ignore")

### Load the dataset

In [6]:
def load_one_dataset():
    import os
    import scipy.io
    import pandas as pd
    
    # Ask user for dataset number
    choice = input("Enter dataset number (1–63): ")
    
    # Convert to integer
    choice = int(choice)
    
    # Base path
    base_dir = os.path.join(os.path.expanduser("~"), "Desktop", "TEAM - Dataset", "data")
    
    # Build file path
    path = os.path.join(base_dir, str(choice), "data.mat")
    
    # Load data
    data = scipy.io.loadmat(path)
    X, y = data['X'], data['y']
    
    # Convert X to DataFrame
    X = pd.DataFrame(
        X,
        columns=[f"feature_{i}" for i in range(X.shape[1])]
    )
    
    print(f"Loaded dataset {choice}")
    print(f"X shape: {X.shape}, y shape: {y.shape}")
    
    return X, y

In [7]:
from sklearn.model_selection import train_test_split

def train_test_split_custom(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y.ravel(),
        test_size=0.2,
        random_state=42
    )
    return X_train, X_test, y_train, y_test

In [8]:
# prediction function
def predict_model(model, X, y):
    # Get predicted probabilities for the positive class (class = 1)
    y_pred_proba_lr = model.predict_proba(X)[:, 1]
    
    # Compute AUC
    auc_score_lr = roc_auc_score(y, y_pred_proba_lr)
    
    return round(auc_score_lr, 4)

In [9]:
def get_rf_shap_values(rf_model, X_train, X):
    # Create explainer
    explainer = shap.Explainer(
        rf_model,
        X_train,
    )
    
    # Compute SHAP values
    shap_values = explainer(X, check_additivity=False)
    shap_class1 = shap_values[:, :, 1]
    
    return shap_class1.values

In [10]:
def get_lime_values(model, X_train, X):
    explainer = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=X_train.columns,
        class_names=["0", "1"],
        mode="classification"
    )
    
    lime_matrix = np.zeros((X.shape[0], X.shape[1]))

    for i in range(X.shape[0]):
        exp = explainer.explain_instance(
            data_row=X.values[i],
            predict_fn=model.predict_proba
        )

        # THIS IS THE FIX ↓
        for idx, weight in exp.as_map()[1]:  # class 1
            lime_matrix[i, idx] = weight

    return lime_matrix

In [11]:
def train_lr_model(X_train, y_train):
    LR_model = LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    )
    LR_model.fit(X_train, y_train)
    return LR_model

In [18]:
def train_rf_model(X_train, y_train):
    RF_model = RandomForestClassifier(n_estimators=100, random_state=42)
    RF_model.fit(X_train, y_train)
    return RF_model

In [12]:
Maybe delete 

def get_lr_shap_values(model, X_train, X):
    explainer = shap.Explainer(model, X_train)
    shap_values = explainer(X, check_additivity=False)

    # Handle both cases automatically
    if len(shap_values.shape) == 3:
        # Tree models (RF, XGBoost, etc.)
        return shap_values[:, :, 1].values
    else:
        # Linear models (Logistic Regression)
        return shap_values.values

In [ ]:
def get_shap_values(model, X_train, X):
    import shap

    explainer = shap.Explainer(model, X_train)
    shap_values = explainer(X, check_additivity=False)

    # Handle both cases
    if len(shap_values.shape) == 3:
        # Tree-based models (RF, etc.)
        return shap_values[:, :, 1].values
    else:
        # Linear models (LR)
        return shap_values.values

In [13]:
def get_person_corr(shap_values, lime_values):
    instance_corrs = []
    p_values = []
    
    for i in range(shap_values.shape[0]):
        shap_i = shap_values[i]
        lime_i = lime_values[i]
        
        corr, p = pearsonr(shap_i, lime_i)
        
        instance_corrs.append(corr)
        p_values.append(p)
    
    instance_corrs = np.array(instance_corrs)
    p_values = np.array(p_values)
    return instance_corrs, p_values

In [14]:
def evaluate_data_subset(X, y, model_fn):
    # split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # train model (generalized)
    model = model_fn(X_train, y_train)
    
    # evaluate
    return predict_model(model, X_test, y_test)

In [29]:
import warnings
warnings.filterwarnings("ignore")

# Using the functions

In [15]:
X, y = load_one_dataset()

Enter dataset number (1–63):  61


Loaded dataset 61
X shape: (38501, 54), y shape: (1, 38501)


In [22]:
X_train, X_test, y_train, y_test = train_test_split_custom(X, y)

In [27]:
rf_model_dataset61 = train_rf_model(X_train, y_train)
# Get predicted probabilities for the positive class (class = 1)
y_pred_proba = rf_model_dataset1.predict_proba(X_test)[:, 1]

# Compute AUC
baseline_rf_auc = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {baseline_rf_auc:.4f}")

AUC: 0.9966


In [ ]:
rf_shap_values61 = get_rf_shap_values(rf_model_dataset61, X_train, X)

In [ ]:
np.save("results/rf_shap_values61.npy", rf_shap_values)

In [ ]:
rf_lime_values61 = get_lime_values(rf_model_dataset1, X_train, X)

In [ ]:
np.save("results/rf_lime_values61.npy", rf_lime_values)

In [ ]:
print("Done")

In [ ]:
lr_model_dataset61 = train_lr_model(X_train, y_train)
y_pred_proba = lf_model_dataset61.predict_proba(X_test)[:, 1]

# Compute AUC
baseline_lf_auc = roc_auc_score(y_test, y_pred_proba)

print(f"AUC: {baseline_rf_auc:.4f}")

In [ ]:
lr_shap_values61 = get_shap_values(lr_model_dataset61, X_train, X)

In [ ]:
np.save("results/lf_shap_values61.npy", lr_shap_values61)

In [ ]:
lr_lime_values61 = get_lime_values(lr_model_dataset61, X_train, X)

In [ ]:
np.save("results/lr_lime_values61.npy", lr_lime_values61)

# Visualisation #

In [ ]:
shap.summary_plot(rf_shap_values61, X)

In [ ]:
lime_importance = np.abs(rf_lime_values).mean(axis=0)

lime_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": lime_importance
}).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(lime_importance_df["feature"], lime_importance_df["importance"])
plt.title("LIME Global Feature Importance for RF Model")
plt.show()

### Correlation


In [ ]:
rf_lime_lr_lime, _ = get_person_corr(rf_lime_values61, lr_lime_values61)
rf_shap_lr_shap, _ = get_person_corr(rf_shap_values61, lr_shap_values61)
rf_lim_rf_shap, _ = get_person_corr(rf_lime_values61, rf_shap_values61)
lr_lim_lr_shap, _ = get_person_corr(lr_lime_values61, lr_shap_values61)

In [ ]:
# Save the results in a dataframe
corr_results_df61 = pd.DataFrame(index=X.index)

corr_results_df["rf_lime_lr_lime"] = rf_lime_lr_lime
corr_results_df["rf_shap_lr_shap"] = rf_shap_lr_shap
corr_results_df["rf_lime_rf_shap"] = rf_lim_rf_shap
corr_results_df["lr_lime_lr_shap"] = lr_lim_lr_shap